In [ ]:
import os
import pandas as pd
from PIL import Image
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import models, transforms
from sklearn.metrics import accuracy_score, roc_auc_score

In [ ]:
import kagglehub
path = kagglehub.dataset_download("bajcsiadel96/romanian-dense-breast-mammography-collection")

In [ ]:
print("Path to dataset files:", path)

base_dir = path

In [ ]:
print(os.listdir(base_dir))

In [ ]:
import os
import pandas as pd
from PIL import Image
import torch
from torch.utils.data import Dataset


class RDBMCPrognosisDataset(Dataset):

  def __init__(self, base_dir, transform=None):
    self.base_dir = base_dir
    self.transform = transform

    csv_path = os.path.join(base_dir, "metadata.csv")
    self.metadata = pd.read_csv(csv_path)


    if "Classification" in self.metadata.columns:
      self.metadata = self.metadata[
          self.metadata["Classification"]
          .astype(str)
          .str.lower()
          .str.contains("malignant")
      ].reset_index(drop=True)

    self.samples = []
    pngs_dir = os.path.join(base_dir, "pngs")

    for _, row in self.metadata.iterrows():
      code = str(row.get("Code", ""))
      img_path = os.path.join(pngs_dir, f"{code}.png")


      if not os.path.exists(img_path):
        matching_files = [
            f for f in os.listdir(pngs_dir) if code in f and f.endswith(".png")
        ]
        if matching_files:
          img_path = os.path.join(pngs_dir, matching_files[0])

      if os.path.exists(img_path):
        self.samples.append({"img_path": img_path, "row": row})

  def __len__(self):
    return len(self.samples)

  def __getitem__(self, idx):
    sample = self.samples[idx]
    row = sample["row"]


    image = Image.open(sample["img_path"]).convert("RGB")
    if self.transform:
      image = self.transform(image)


    age_val = row.get("Age", 50.0)
    if pd.isna(age_val):
      age_val = 50.0

    grade_val = row.get("Nottingham classification_grade", 2.0)
    if pd.isna(grade_val):
      grade_val = 2.0

    tabular_tensor = torch.tensor(
        [float(age_val), float(grade_val)], dtype=torch.float32
    )
    tabular_tensor = torch.nan_to_num(tabular_tensor, nan=0.0)


    cancer_type = str(row.get("Cancer type", "")).lower()
    associated_features = str(row.get("Mass_Associated features", "")).lower()

    if (
        "metastasis" in cancer_type
        or "axillary adenopathy" in associated_features
    ):
      target_val = 1.0
    else:
      target_val = 0.0

    label = torch.tensor(target_val, dtype=torch.float32)

    return image, tabular_tensor, label

In [ ]:
class CrossAttentionFusionModel(nn.Module):
    def __init__(self, model_name="efficientnet_b0", num_tabular_features=2, embed_dim=128, num_heads=4):
        super(CrossAttentionFusionModel, self).__init__()


        self.image_encoder = timm.create_model(model_name, pretrained=True, num_classes=0)
        img_feature_dim = self.image_encoder.num_features
        self.img_projection = nn.Linear(img_feature_dim, embed_dim)


        self.tabular_encoder = nn.Sequential(
            nn.Linear(num_tabular_features, 64),
            nn.ReLU(),
            nn.Linear(64, embed_dim),
            nn.ReLU()
        )


        self.cross_attention = nn.MultiheadAttention(embed_dim=embed_dim, num_heads=num_heads, batch_first=True)
        self.layer_norm = nn.LayerNorm(embed_dim)


        self.classifier = nn.Sequential(
            nn.Linear(embed_dim * 2, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 1)
        )

    def forward(self, images, tabular):
        img_feats = self.img_projection(self.image_encoder(images))
        tab_feats = self.tabular_encoder(tabular)

        img_query = img_feats.unsqueeze(1)
        tab_key_value = tab_feats.unsqueeze(1)


        attn_output, _ = self.cross_attention(query=img_query, key=tab_key_value, value=tab_key_value)
        attn_output = self.layer_norm(img_query + attn_output).squeeze(1)


        fused = torch.cat((attn_output, tab_feats), dim=1)

        logits = self.classifier(fused)
        return logits.squeeze(1)

In [ ]:

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])


dataset = RDBMCPrognosisDataset(base_dir=base_dir, transform=transform)
print(f"Total malignant samples loaded: {len(dataset)}")

if len(dataset) > 0:
    train_size = int(0.8 * len(dataset))
    test_size = len(dataset) - train_size
    train_dataset, test_dataset = random_split(dataset, [train_size, test_size])

    train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = CrossAttentionFusionModel(model_name="efficientnet_b0", num_tabular_features=2).to(device)

    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-4)

    print(f"Dataset loaded successfully with {len(dataset)} samples. Starting training...")


    for epoch in range(3):
        model.train()
        total_train_loss = 0
        for images, tabular, labels in train_loader:
            images, tabular, labels = images.to(device), tabular.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images, tabular)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            total_train_loss += loss.item()


        model.eval()
        total_test_loss = 0
        correct = 0
        total = 0
        with torch.no_grad():
            for images, tabular, labels in test_loader:
                images, tabular, labels = images.to(device), tabular.to(device), labels.to(device)
                outputs = model(images, tabular)
                loss = criterion(outputs, labels)
                total_test_loss += loss.item()

                preds = (torch.sigmoid(outputs) >= 0.5).float()
                correct += (preds == labels).sum().item()
                total += labels.size(0)

        acc = (correct / total) * 100 if total > 0 else 0
        print(f"--- Epoch {epoch+1}/3 ---")
        print(f"Train Loss: {total_train_loss/len(train_loader):.4f}")
        print(f"Test Loss: {total_test_loss/len(test_loader):.4f} | Test Accuracy: {acc:.2f}%")
else:
    print("Warning: Dataset loaded 0 samples. Check metadata columns or image paths.")

In [ ]:

all_labels = []
for _, _, label in dataset:
  all_labels.append(label.item())

import pandas as pd

label_counts = pd.Series(all_labels).value_counts()
print("Label Distribution (0 = No Metastasis/Adenopathy, 1 = Yes):")
print(label_counts)

In [ ]:
meta = pd.read_csv(os.path.join(base_dir, "metadata.csv"))
print(meta.columns.tolist())
print(meta["Classification"].value_counts(dropna=False))
print(meta["Cancer type"].value_counts(dropna=False))
print(meta["Mass_Associated features"].value_counts(dropna=False))

for c in meta.columns:
    if any(t in c.lower() for t in ["metastasis", "adenopathy", "node"]):
        print(f"\n{c}:")
        print(meta[c].astype(str).value_counts(dropna=False))

In [ ]:
import os
import pandas as pd


csv_path = os.path.join(path, "metadata.csv")
df_meta = pd.read_csv(csv_path)


if "Classification" in df_meta.columns:
  df_malignant = df_meta[
      df_meta["Classification"].astype(str).str.lower().str.contains("malignant")
  ].copy()
else:
  df_malignant = df_meta.copy()

total_malignant = len(df_malignant)


def get_target(row):
  cancer_type = str(row.get("Cancer type", "")).lower()
  associated_features = str(row.get("Mass_Associated features", "")).lower()

  if (
      "metastasis" in cancer_type
      or "axillary adenopathy" in associated_features
  ):
    return 1.0
  return 0.0


df_malignant["Target"] = df_malignant.apply(get_target, axis=1)

pos_count = (df_malignant["Target"] == 1.0).sum()
neg_count = (df_malignant["Target"] == 0.0).sum()

print(f"Total Malignant Patients: {total_malignant}")
print(f"Target = 1.0 (Metastasis / Axillary Adenopathy): {pos_count}")
print(f"Target = 0.0 (Otherwise / Localized): {neg_count}")